# 🌱 CSIRO Image2Biomass - Complete Solution (No GitHub Required)

**Multi-Modal Deep Learning Pipeline - Self-Contained Version**

⚙️ **Notebook Settings**:
- Accelerator: **GPU T4 x2** or **GPU P100**
- Internet: **ON** (for pip install only)
- Persistence: Files only

## 📥 Step 1: Install Dependencies

In [ ]:
%%time
# Install required packages with compatible versions
!pip install -q timm==0.9.12
!pip install -q albumentations==1.3.1 --no-deps
!pip install -q albucore==0.0.17 qudida==0.0.4

import torch
print(f"\n✅ Installation complete!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 🔧 Step 2: Create Project Structure

In [ ]:
import os

# Create directories
os.makedirs('/kaggle/working/src', exist_ok=True)
os.makedirs('/kaggle/working/configs', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/submissions', exist_ok=True)

print("✅ Directories created")

## 📝 Step 3: Write Source Code Files

In [ ]:
%%writefile /kaggle/working/configs/config.yaml
# CSIRO Image2Biomass Configuration
data:
  data_dir: "/kaggle/input/csiro-biomass"
  train_csv: "train.csv"
  test_csv: "test.csv"
  image_dir: "images"
  img_size: 384
  n_folds: 5
  seed: 42

model:
  architectures:
    - efficientnet_b3
  pretrained: true
  num_classes: 1
  dropout: 0.3
  use_ndvi: true
  use_metadata: true
  metadata_features:
    - season
    - region

training:
  batch_size: 16
  num_epochs: 15
  learning_rate: 0.0003
  weight_decay: 0.0001
  scheduler: "CosineAnnealingLR"
  mixed_precision: true
  accumulation_steps: 2
  patience: 7
  gradient_clip: 1.0
  loss_fn: "smooth_l1"

augmentation:
  train:
    horizontal_flip: 0.5
    vertical_flip: 0.5
    rotate_limit: 30
    brightness_limit: 0.2
    contrast_limit: 0.2

inference:
  use_tta: true
  tta_transforms: 4
  batch_size: 32
  tta_merge_mode: "mean"

logging:
  use_wandb: false

paths:
  checkpoint_dir: "/kaggle/working/checkpoints"
  best_model_dir: "/kaggle/working/best_models"
  submission_dir: "/kaggle/working/submissions"

### Write Utils Module

In [ ]:
%%writefile /kaggle/working/src/utils.py
import os
import random
import numpy as np
import torch

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif self.mode == 'min':
            if score < self.best_score - self.min_delta:
                self.best_score = score
                self.counter = 0
            else:
                self.counter += 1
        if self.counter >= self.patience:
            self.early_stop = True

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

### Write Dataset Module

In [ ]:
%%writefile /kaggle/working/src/dataset.py
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class BiomassDataset(Dataset):
    def __init__(self, df, image_dir, img_size=384, transform=None, 
                 use_ndvi=True, use_metadata=True, is_training=True):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.img_size = img_size
        self.transform = transform
        self.use_ndvi = use_ndvi
        self.use_metadata = use_metadata
        self.is_training = is_training
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        img_path = os.path.join(self.image_dir, row.get('image_path', f"{row['id']}.jpg"))
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image=image)['image']
        else:
            image = cv2.resize(image, (self.img_size, self.img_size))
            image = torch.from_numpy(image.transpose(2,0,1)).float() / 255.0
        
        sample = {'image': image, 'id': row.get('id', idx)}
        
        if self.use_ndvi and 'ndvi' in row:
            sample['ndvi'] = torch.tensor([row['ndvi']], dtype=torch.float32)
        
        if self.is_training and 'target' in row:
            sample['target'] = torch.tensor([row['target']], dtype=torch.float32)
        elif self.is_training and 'biomass' in row:
            sample['target'] = torch.tensor([row['biomass']], dtype=torch.float32)
        
        return sample

def get_transforms(img_size, is_train=True):
    if is_train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Rotate(limit=30, p=0.7),
            A.RandomBrightnessContrast(p=0.7),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])

### Write Model Module

In [ ]:
%%writefile /kaggle/working/src/models.py
import torch
import torch.nn as nn
import timm

class BiomassModel(nn.Module):
    def __init__(self, model_name='efficientnet_b3', pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, 
                                         num_classes=0, global_pool='avg')
        
        # Get feature dimension
        with torch.no_grad():
            dummy = torch.randn(1, 3, 224, 224)
            features = self.backbone(dummy)
            feat_dim = features.shape[1]
        
        # Head
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
    
    def forward(self, batch):
        x = batch['image']
        features = self.backbone(x)
        output = self.head(features)
        return output

## 🏋️ Step 4: Training Pipeline

In [ ]:
import sys
sys.path.append('/kaggle/working/src')

import yaml
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import KFold
from tqdm import tqdm
import numpy as np

from utils import seed_everything, AverageMeter, EarlyStopping
from dataset import BiomassDataset, get_transforms
from models import BiomassModel

# Load config
with open('/kaggle/working/configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

seed_everything(config['data']['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

In [ ]:
# Load data
train_df = pd.read_csv(f"{config['data']['data_dir']}/train.csv")
print(f"Loaded {len(train_df)} training samples")
print(f"Columns: {list(train_df.columns)}")
print(train_df.head())

In [ ]:
%%time
# Training function
def train_fold(fold, train_idx, valid_idx):
    print(f"\n{'='*50}")
    print(f"Training Fold {fold}")
    print(f"{'='*50}\n")
    
    # Create datasets
    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    valid_data = train_df.iloc[valid_idx].reset_index(drop=True)
    
    train_dataset = BiomassDataset(
        train_data, 
        f"{config['data']['data_dir']}/{config['data']['image_dir']}",
        img_size=config['data']['img_size'],
        transform=get_transforms(config['data']['img_size'], True),
        is_training=True
    )
    
    valid_dataset = BiomassDataset(
        valid_data,
        f"{config['data']['data_dir']}/{config['data']['image_dir']}",
        img_size=config['data']['img_size'],
        transform=get_transforms(config['data']['img_size'], False),
        is_training=True
    )
    
    train_loader = DataLoader(train_dataset, batch_size=config['training']['batch_size'],
                            shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=config['training']['batch_size'],
                            shuffle=False, num_workers=2, pin_memory=True)
    
    # Model
    model = BiomassModel(
        model_name=config['model']['architectures'][0],
        pretrained=config['model']['pretrained'],
        dropout=config['model']['dropout']
    ).to(device)
    
    # Optimizer & Scheduler
    optimizer = AdamW(model.parameters(), 
                     lr=config['training']['learning_rate'],
                     weight_decay=config['training']['weight_decay'])
    scheduler = CosineAnnealingLR(optimizer, T_max=config['training']['num_epochs'])
    
    criterion = nn.SmoothL1Loss()
    scaler = torch.cuda.amp.GradScaler()
    early_stopping = EarlyStopping(patience=config['training']['patience'])
    
    best_rmse = float('inf')
    
    # Training loop
    for epoch in range(config['training']['num_epochs']):
        # Train
        model.train()
        losses = AverageMeter()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['training']['num_epochs']}")
        for batch in pbar:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in batch.items()}
            
            with torch.cuda.amp.autocast():
                preds = model(batch)
                loss = criterion(preds, batch['target'])
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            losses.update(loss.item(), batch['target'].size(0))
            pbar.set_postfix({'loss': losses.avg})
        
        # Validate
        model.eval()
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for batch in valid_loader:
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                        for k, v in batch.items()}
                preds = model(batch)
                val_preds.append(preds.cpu().numpy())
                val_targets.append(batch['target'].cpu().numpy())
        
        val_preds = np.concatenate(val_preds)
        val_targets = np.concatenate(val_targets)
        rmse = np.sqrt(np.mean((val_preds - val_targets)**2))
        
        print(f"Epoch {epoch+1}: Train Loss={losses.avg:.4f}, Valid RMSE={rmse:.4f}")
        
        scheduler.step()
        
        # Save best
        if rmse < best_rmse:
            best_rmse = rmse
            torch.save({
                'model': model.state_dict(),
                'rmse': rmse,
                'epoch': epoch
            }, f"/kaggle/working/checkpoints/best_fold{fold}.pth")
            print(f"✅ Best model saved! RMSE: {rmse:.4f}")
        
        early_stopping(rmse)
        if early_stopping.early_stop:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    return best_rmse, model

# Run training
kfold = KFold(n_splits=config['data']['n_folds'], shuffle=True, 
              random_state=config['data']['seed'])

fold_scores = []
for fold, (train_idx, valid_idx) in enumerate(kfold.split(train_df)):
    score, _ = train_fold(fold, train_idx, valid_idx)
    fold_scores.append(score)
    print(f"\nFold {fold} Best RMSE: {score:.4f}")

print(f"\n{'='*50}")
print(f"Mean CV RMSE: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"{'='*50}")

## 🔮 Step 5: Generate Predictions

In [ ]:
# Load test data
test_df = pd.read_csv(f"{config['data']['data_dir']}/test.csv")
print(f"Test samples: {len(test_df)}")

test_dataset = BiomassDataset(
    test_df,
    f"{config['data']['data_dir']}/{config['data']['image_dir']}",
    img_size=config['data']['img_size'],
    transform=get_transforms(config['data']['img_size'], False),
    is_training=False
)

test_loader = DataLoader(test_dataset, batch_size=32, 
                        shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
%%time
# Inference with ensemble
all_predictions = []

for fold in range(config['data']['n_folds']):
    checkpoint_path = f"/kaggle/working/checkpoints/best_fold{fold}.pth"
    if not os.path.exists(checkpoint_path):
        continue
    
    print(f"Loading fold {fold}...")
    model = BiomassModel(
        model_name=config['model']['architectures'][0],
        pretrained=False
    ).to(device)
    
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Fold {fold}"):
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in batch.items()}
            preds = model(batch)
            fold_preds.append(preds.cpu().numpy())
    
    fold_preds = np.concatenate(fold_preds)
    all_predictions.append(fold_preds)

# Ensemble
final_predictions = np.mean(all_predictions, axis=0).squeeze()
print(f"\n✅ Predictions generated for {len(final_predictions)} samples")

In [ ]:
# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'biomass': final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print("\n✅ Submission created!")
print("\nSubmission preview:")
print(submission.head(10))
print(f"\nStatistics:")
print(submission['biomass'].describe())
print(f"\n💾 Saved to: /kaggle/working/submission.csv")

## ✅ Done!

Download `submission.csv` from the Output tab and submit to the competition!